## Load and Compute

In [ ]:
from metrics import ADI
import numpy as np
import pandas as pd
from project_consts import DATASET_DIR, EMBEDDINGS_DIR, RESULTS_DIR
from tqdm import tqdm
import json

In [ ]:
with open(RESULTS_DIR / "cluster_res.json", 'r') as f:
    data = json.load(f)

In [ ]:
cadi_scores = {
    dataset: {
        alg: alg_data['CADI']
        for alg, alg_data in
        ds_data.items()
    }
    for dataset, ds_data
    in data.items()
}

In [ ]:
cadi_df = pd.DataFrame(cadi_scores).T

In [ ]:
seed = 100

scores = dict()

dsfiles = list(DATASET_DIR.iterdir())
for dsfile in tqdm(dsfiles, position=0):
    dataset_name = dsfile.stem
    X = np.load(dsfile)
    embs = {
        embfile.stem : np.load(embfile)
        for embfile in (EMBEDDINGS_DIR / dataset_name).iterdir()
    }

    scores[dataset_name] = {embname: ADI(X, Y, X.shape[0] * 100, random_seed=100) for embname, Y in tqdm(embs.items(), desc=dataset_name, position=1, leave=False)}

100%|██████████| 19/19 [04:29<00:00, 14.19s/it]


In [ ]:
adi_df = pd.DataFrame(scores).T

In [ ]:
adi_df.style.background_gradient(axis=1, cmap='PuBu_r')

,MDS,PCA,PaCMap,Random,TSNE,UMAP,UMATO,AngleEmbedding
MNIST,0.083053,0.089938,0.088891,0.101454,0.088425,0.099534,0.092745,0.103698
acl_imdb,0.090370,0.098507,0.100723,0.098311,0.094213,0.100611,0.101444,0.091998
coil100,0.057479,0.060339,0.098790,0.109955,0.078859,0.109839,0.074145,0.066167
coil20,0.062430,0.077140,0.086193,0.106856,0.078956,0.103771,0.083444,0.079969
concentric3,0.028746,0.031440,0.198317,0.180978,0.159908,0.162552,0.043313,0.027078
concentric4,0.033928,0.032311,0.136832,0.169770,0.139049,0.152120,0.046598,0.028034
donuts,0.002692,0.002692,0.091151,0.170573,0.054210,0.081013,0.011219,0.002705
emotion,0.089980,0.096158,0.097051,0.098204,0.095090,0.097323,0.094331,0.106154
fashionMNIST,0.051249,0.052381,0.069534,0.110581,0.069923,0.082302,0.067551,0.079502
liver,0.061103,0.070559,0.112603,0.109620,0.077614,0.087484,0.083816,0.064973


## CADI vs ADI

In [ ]:
corr_df = adi_df.corrwith(cadi_df, axis=1, method='pearson')

In [ ]:
corr_df

MNIST           0.072793
acl_imdb        0.583305
coil100         0.517594
coil20          0.557672
concentric3     0.979592
concentric4     0.991507
donuts          0.991830
emotion        -0.741660
fashionMNIST    0.780169
liver           0.740279
matryoshka      0.914035
olivetti        0.697094
pbmc3k         -0.765146
pendigits       0.913457
penguins        0.616402
rings           0.778213
sentiment       0.792202
trec           -0.209359
usps            0.568312
dtype: float64

## ADI is global

In [ ]:
sorted_idx = np.argsort(adi_df.to_numpy(), axis=1)

bests = pd.DataFrame({
    'Rank 1': adi_df.columns[sorted_idx[:, 0]],
    'Rank 2': adi_df.columns[sorted_idx[:, 1]]
}, index=adi_df.index)

bests

,Rank 1,Rank 2
MNIST,MDS,TSNE
acl_imdb,MDS,AngleEmbedding
coil100,MDS,PCA
coil20,MDS,PCA
concentric3,AngleEmbedding,MDS
concentric4,AngleEmbedding,PCA
donuts,MDS,PCA
emotion,MDS,UMATO
fashionMNIST,MDS,PCA
liver,MDS,AngleEmbedding


In [ ]:
(bests == 'MDS').sum()

Rank 1    13
Rank 2     5
dtype: int64

In [ ]:
(bests == 'PCA').sum()

Rank 1    1
Rank 2    8
dtype: int64

### CADI bests

In [ ]:
sorted_idx = np.argsort(cadi_df.to_numpy(), axis=1)

cadi_bests = pd.DataFrame({
    'Best': cadi_df.columns[sorted_idx[:, 0]],
    '2nd Best': cadi_df.columns[sorted_idx[:, 1]]
}, index=cadi_df.index)

cadi_bests

,Best,2nd Best
usps,AngleEmbedding,TSNE
trec,AngleEmbedding,PCA
sentiment,AngleEmbedding,MDS
rings,AngleEmbedding,MDS
penguins,AngleEmbedding,TSNE
pendigits,AngleEmbedding,UMATO
pbmc3k,AngleEmbedding,TSNE
olivetti,AngleEmbedding,UMAP
matryoshka,AngleEmbedding,PCA
liver,AngleEmbedding,PCA


In [ ]:
(cadi_bests == 'MDS').sum()

Best        1
2nd Best    4
dtype: int64

In [ ]:
(cadi_bests == 'PCA').sum()

Best        0
2nd Best    4
dtype: int64